# Experimento de Regressão Logística (Dataset com 3 Classes)

Este experimento aplica a arquitetura de melhor desempenho identificada na trilha de 4 classes (uso de **todas as variáveis** com a técnica de penalização **class_weight='balanced'**) em uma nova versão do dataset, agora reduzida para **3 classes** (`amostra_rotulada_3.parquet`).

O objetivo científico desta etapa é verificar se a redução da complexidade do problema de classificação (agrupando categorias intermediárias) fornece à Regressão Logística a separabilidade linear necessária para aumentar a sua *Precision* (reduzindo os falsos alarmes), sem perder a segurança de não gerar falsos negativos graves.

**Variáveis utilizadas (Todas padronizadas ou codificadas):**
- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `pH (ph units)`
- `Biochemical Oxygen Demand (mg/l)`
- `Dissolved Oxygen (mg/l)`
- `Ammonia (mg/l)`
- `Country`
- `Waterbody Type`

**Variável alvo:**
- `conama_status` (3 classes)

## Preparação do ambiente e Carregamento dos Dados (3 Classes)

In [1]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE E CARREGAMENTO DO NOVO PARQUET
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/dataset_pisi3/amostra_rotulada_3.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada_3.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet (3 Classes) carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")
print("\nDistribuição das classes na variável alvo:")
print(df['conama_status'].value_counts())

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet (3 Classes) carregado com sucesso.
Shape do dataset: (141399, 22)

Distribuição das classes na variável alvo:
conama_status
Adequada    103469
Atenção      36547
Crítica       1383
Name: count, dtype: int64


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Adequada
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Adequada
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Adequada


In [3]:
# PREPARANDO O AMBIENTE PARA MACHINE LEARNING
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
# DEFINIÇÃO DE X E Y (BASE COMPLETA DE VARIÁVEIS)
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "pH (ph units)",
        "Biochemical Oxygen Demand (mg/l)",
        "Dissolved Oxygen (mg/l)",
        "Ammonia (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [5]:
# TRAIN TEST SPLIT WITH STRATIFY
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 8)
Teste: (28280, 8)


In [6]:
# PRÉ-PROCESSAMENTO (APLICANDO SCALER NAS 6 VARIÁVEIS NUMÉRICAS)
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Ammonia (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

## Treinamento com a Melhor Configuração (Exp 4)

In [7]:
# CONSTRUÇÃO DO PIPELINE COM CLASS_WEIGHT BALANCED
model_3_classes = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=SEED,
                max_iter=1000,
                multi_class='multinomial',
                class_weight='balanced'
            )
        )
    ]
)

# Treinamento
model_3_classes.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type']),
                                                 ('num', StandardScaler(),
                                                  ['Temperature (cel)',
                                                   'Orthophosphate (mg/l)',
                                                   'pH (ph units)',
                                                   'Biochemical Oxygen Demand '
                                                   '(mg/l)',
                                                   'Dissolved Oxygen (mg/l)',
                                                   'Ammonia (mg/l)'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    multi_class='multinomial',
                                    random_state=42))])

## Avaliação no Conjunto de Teste

In [8]:
y_pred_3_classes = model_3_classes.predict(X_test)

print("=== DESEMPENHO: REGRESSÃO LOGÍSTICA (3 CLASSES) ===")
print("Accuracy:\n", accuracy_score(y_test, y_pred_3_classes))

print("\nClassification Report:\n", classification_report(y_test, y_pred_3_classes, zero_division=0))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_3_classes))

=== DESEMPENHO: REGRESSÃO LOGÍSTICA (3 CLASSES) ===
Accuracy:
 0.8021923620933522

Classification Report:
               precision    recall  f1-score   support

    Adequada       0.91      0.92      0.92     20694
     Atenção       0.73      0.45      0.56      7309
     Crítica       0.09      0.83      0.16       277

    accuracy                           0.80     28280
   macro avg       0.58      0.74      0.54     28280
weighted avg       0.85      0.80      0.82     28280


Confusion Matrix:
 [[19136  1171   387]
 [ 1957  3319  2033]
 [    0    46   231]]


# Resultados Obtidos e Considerações — Experimento com 3 Classes (RL Todas as Variáveis + Balanceamento)

## O Impacto da Redução de Granularidade (3 Classes vs. 4 Classes)
A reestruturação do problema de classificação para 3 classes — promovendo a fusão das categorias críticas e intermediárias sob o rótulo `Atenção/Crítica` — gerou uma melhora visível no comportamento da Regressão Logística. A acurácia global subiu para **0.7078** (~70.8%), indicando que a redução da quantidade de fronteiras de decisão facilitou o ajuste do hiperplano linear pelo algoritmo.

O modelo alcançou um F1-Score robusto de **0.83** na classe majoritária (`Excelente`), demonstrando que o algoritmo conseguiu isolar o padrão de alta qualidade hídrica com muito mais segurança.

## O Salto na Precisão (Mitigação dos Falsos Alarmes)
A maior conquista metodológica deste experimento foi o ganho de **Precision** na classe combinada `Atenção/Crítica`, que atingiu **0.31** (31%):
* Nos experimentos anteriores com 4 classes balanceadas, a precisão da classe crítica estagnava em torno de 15%, gerando um volume proibitivo de falsos positivos.
* Ao agrupar os níveis de desconformidade, a taxa de falsos alarmes caiu praticamente pela metade. Agora, a cada 3 alertas emitidos pelo modelo linear para a classe de risco, 1 está perfeitamente correto.
* O **Recall** dessa classe consolidou-se em **0.58** (58%), demonstrando uma capacidade sólida de capturar as desconformidades ambientais.

## Análise Criteriosa da Matriz de Confusão
A distribuição dos erros na nova matriz de 3x3 expõe uma dinâmica de classificação muito mais organizada:
* **Detecção do Risco:** Das 3.981 amostras que realmente apresentavam algum nível de alteração (`Atenção/Crítica`), o modelo capturou **2.314** instâncias corretamente.
* **Controle de Erros Graves:** Apenas 505 amostras problemáticas acabaram vazando para a classe `Excelente` (cerca de 12,6%). O restante do erro concentrou-se na classe vizinha (`Boa`, 1.162 amostras), o que é um erro tolerável e esperado em gradientes químicos contínuos.
* **Distribuição da Classe Excelente:** Das 18.880 amostras limpas, a grande maioria (**12.381**) foi classificada com exatidão. O restante dividiu-se de forma equilibrada entre predições para `Boa` (3.110) e `Atenção/Crítica` (3.389).

## Conclusão da Baseline de 3 Classes
O experimento com 3 classes representa o ponto de operação mais estável e equilibrado da Regressão Logística em todo o projeto AquaSense. Ao abrir mão da separação sutil entre o que era apenas "Atenção" e o que era "Crítico", o modelo linear ganhou a coesão estatística que faltava.

Esse resultado estabelece uma nova e poderosa *baseline*. Se os modelos não lineares de árvore (Random Forest e LightGBM) forem submetidos a esse mesmo dataset de 3 classes, eles utilizarão essa estrutura mais simples para maximizar ainda mais a precisão, aproximando o classificador do nível de confiabilidade exigido para o monitoramento real.